# Inference LLM on LAN

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

from fastapi import FastAPI
from pydantic import BaseModel, Field
import uvicorn
import torch
import nest_asyncio
import socket
import time
import uuid
from typing import Literal

In [4]:
hf_token = ""
# If needed: login(token=hf_token)

In [5]:
model_name = "quangne/text2diagram-AceMath-1.5B-Instruct-merged-geometry3k8-8-1-1"

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

use_cuda = torch.cuda.is_available()
load_dtype = (
    torch.bfloat16 if (use_cuda and torch.cuda.is_bf16_supported())
    else (torch.float16 if use_cuda else torch.float32)
 )

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=load_dtype,
    device_map="auto"
 )
model.eval()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:35<00:00,  9.60it/s]
Some parameters are on the meta device because they were offloaded to the cpu and disk.


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [6]:
nest_asyncio.apply()

In [7]:
def prepare_inference_context(model):
    model.eval()
    use_cuda = torch.cuda.is_available() and str(model.device).startswith("cuda")
    compute_dtype = torch.bfloat16 if (use_cuda and torch.cuda.is_bf16_supported()) else torch.float16
    return use_cuda, compute_dtype

In [8]:
app = FastAPI()

class ChatMessage(BaseModel):
    role: Literal["system", "user", "assistant"]
    content: str


class ChatCompletionsRequest(BaseModel):
    model: str = "acemath-mock"
    messages: list[ChatMessage] = Field(default_factory=list)
    max_tokens: int = 256
    temperature: float = 0.0
    top_p: float = 1.0
    repetition_penalty: float = 1.08


def build_prompt(messages: list[ChatMessage]) -> str:
    # Combine messages into a single prompt string
    lines = []
    for m in messages:
        if m.role == "system":
            lines.append(f"[System]\n{m.content}")
        elif m.role == "user":
            lines.append(f"[User]\n{m.content}")
        else:
            lines.append(f"[Assistant]\n{m.content}")
    return "\n\n".join(lines).strip()


def generate_dsl(
    prompt_text: str,
    max_new_tokens: int = 256,
    use_cuda: bool = False,
    compute_dtype: torch.dtype = torch.float16,
) -> str:
    messages = [{"role": "user", "content": prompt_text}]
    if tokenizer.chat_template:
        rendered_prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    else:
        rendered_prompt = f"User:\n{prompt_text}\n\nAssistant:\n"

    inputs = tokenizer(rendered_prompt, return_tensors="pt").to(model.device)

    with torch.inference_mode():
        if use_cuda:
            with torch.autocast(device_type="cuda", dtype=compute_dtype):
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    repetition_penalty=1.08,
                    eos_token_id=tokenizer.eos_token_id,
                    pad_token_id=tokenizer.pad_token_id,
                    use_cache=True,
                )
        else:
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                repetition_penalty=1.08,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id,
                use_cache=True,
                )

    prompt_len = inputs["input_ids"].shape[-1]
    generated = outputs[0][prompt_len:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


@app.post("/v1/chat/completions")
def chat_completions(req: ChatCompletionsRequest):
    prompt_text = build_prompt(req.messages)
    if not prompt_text:
        prompt_text = ""

    use_cuda, compute_dtype = prepare_inference_context(model)
    text = generate_dsl(
        prompt_text=prompt_text,
        max_new_tokens=req.max_tokens,
        use_cuda=use_cuda,
        compute_dtype=compute_dtype,
    )

    return {
        "id": f"chatcmpl-{uuid.uuid4().hex}",
        "object": "chat.completion",
        "created": int(time.time()),
        "model": req.model,
        "choices": [
            {
                "index": 0,
                "message": {
                    "role": "assistant",
                    "content": text,
                },
                "finish_reason": "stop",
            }
        ],
        "usage": {
            "prompt_tokens": 0,
            "completion_tokens": 0,
            "total_tokens": 0,
        },
    }

In [ ]:
HOST = "0.0.0.0"
PORT = 8001

def get_lan_ip():
    s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    try:
        s.connect(("8.8.8.8", 80))
        ip = s.getsockname()[0]
    except Exception:
        ip = "127.0.0.1"
    finally:
        s.close()
    return ip

lan_ip = get_lan_ip()
print("LAN URL:", f"http://{lan_ip}:{PORT}/v1/chat/completions")

server = uvicorn.Server(uvicorn.Config(app, host=HOST, port=PORT))
await server.serve()

LAN URL: http://192.168.1.17:8000/v1/chat/completions


INFO:     Started server process [13928]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [13928]
